# Install and load necesary packages

In [1]:
# Please don't change this cell

import pandas as pd
import numpy as np  

import warnings
warnings.filterwarnings("ignore")

## Load the dataset using pandas

In [2]:
# Please don't change this cell
df = pd.read_csv('ml-100k/u.data', names=['user_id', 'item_id', 'rating', 'timestamp'], sep='\t')

df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


# Split dataset


## Randomly select one rating from each user as test set

In [3]:
# please do not change this cell

from sklearn.model_selection import train_test_split

n_users = df.user_id.unique().shape[0]
n_items = df.item_id.unique().shape[0]
print(str(n_users) + ' users')
print(str(n_items) + ' items')

train_df, test_df = train_test_split(df, test_size=0.2, random_state = 10)
train_df, test_df

# Training Dataset
train_ds = np.zeros((n_users, n_items))
item_popularity = np.zeros(n_items)
for row in train_df.itertuples():
    train_ds[row[1]-1, row[2]-1] = row[3]
    item_popularity[row[2]-1] =  item_popularity[row[2]-1] + 1
#train_ds = pd.DataFrame(train_ds)

# Testing Dataset
testsize = 0
test_ds = np.zeros((n_users, n_items))
for row in test_df.itertuples():
    if item_popularity[row[2]-1] > 30:
        test_ds[row[1]-1, row[2]-1] = row[3]
        testsize = testsize + 1
#test_ds = pd.DataFrame(test_ds)

print("Construct the rating matrix based on train_df:")
print(train_ds)

print("Construct the rating matrix based on test_df:")
print(test_ds)

print("Testsize = " + str(testsize))

943 users
1682 items
Construct the rating matrix based on train_df:
[[0. 3. 4. ... 0. 0. 0.]
 [4. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 5. 0. ... 0. 0. 0.]]
Construct the rating matrix based on test_df:
[[5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Testsize = 17678


# MAE and RMSE Utils

In [4]:
# Please don't change this cell
# you can use this devaluate Utils here, and you can also implement your own MAE and RMSE calculation. 

EPSILON = 1e-9

def evaluate(test_ds, predicted_ds):
    '''
    Function for evaluating on MAE and RMSE
    '''
    # MAE
    mask_test_ds = test_ds > 0
    MAE = np.sum(np.abs(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32))

    # RMSE
    RMSE = np.sqrt(np.sum(np.square(test_ds[mask_test_ds] - predicted_ds[mask_test_ds])) / np.sum(mask_test_ds.astype(np.float32)))

    return MAE, RMSE

# Your Solution for Method 1

In [5]:
# Write your code here for Method 1
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution1 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution1 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.

# --------------------------
# Method 1: User Mean
# --------------------------

# Step 1: Compute each user's mean rating from the training matrix
# Explanation:
# - For each user 'u', calculate the average of all non-zero ratings (i.e., rated items).
# - Avoid division by zero by adding a small constant (EPSILON) to the denominator.
# - If a user hasn't rated any items, assign a default mean of 0.0.

# Count of rated items per user (shape: [n_users])
user_rated_counts = (train_ds > 0).sum(axis=1)

# Sum of all ratings per user (shape: [n_users])
user_sum_ratings = train_ds.sum(axis=1)

# Compute mean rating per user with safety against divide-by-zero
user_means = np.where(
    user_rated_counts > 0,
    user_sum_ratings / (user_rated_counts + EPSILON),
    0.0
)

# Step 2: Create the prediction matrix using user means
# Explanation:
# - Initialize an empty prediction matrix with the same shape as train_ds.
# - For every user 'u', fill all item columns with their average rating.
# - This assumes the user gives the same rating to all items when no info is available.

# Initialize prediction matrix (shape: [n_users, n_items])
predicted_ds = np.zeros_like(train_ds)

# Fill each row 'u' with the corresponding user's mean rating
for u in range(n_users):
    predicted_ds[u, :] = user_means[u]

# Step 3: Evaluate the predictions on the test dataset
# Explanation:
# - Use the provided evaluate() function to calculate Mean Absolute Error (MAE)
#   and Root Mean Square Error (RMSE) between test ratings and predictions.

MAE_solution1, RMSE_solution1 = evaluate(test_ds, predicted_ds)

# The results MAE_solution1 and RMSE_solution1 represent the performance
# of the User Mean baseline model.



In [6]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution1, RMSE_solution1))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.8258905090160901, RMSE: 1.0311430705959619


# Your Solution for Method 2

In [7]:
# Write your code here for Method 2
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution2 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution2 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.


# --------------------------
# Method 2: Item Mean
# --------------------------

# Step 1: Compute each item's mean rating from the training matrix
# Explanation:
# - For each item 'i', compute the average of all ratings received from users.
# - Use a small EPSILON in the denominator to avoid division by zero for items with no ratings.
# - If an item has not been rated by any user, set its mean to 0.0.

# Count how many users rated each item (shape: [n_items])
item_rated_counts = (train_ds > 0).sum(axis=0)

# Total sum of ratings for each item (shape: [n_items])
item_sum_ratings = train_ds.sum(axis=0)

# Compute mean rating per item, handling division by zero using EPSILON
item_means = np.where(
    item_rated_counts > 0,
    item_sum_ratings / (item_rated_counts + EPSILON),
    0.0
)

# Step 2: Create a prediction matrix using item means
# Explanation:
# - Initialize a matrix with the same shape as the training matrix.
# - Fill every column 'i' with the mean rating of item 'i'.
# - This assumes every user would rate item 'i' similarly, based on its popularity.

# Initialize prediction matrix (shape: [n_users, n_items])
predicted_ds2 = np.zeros_like(train_ds)

# For each item, fill its column with its mean rating
for i in range(n_items):
    predicted_ds2[:, i] = item_means[i]

# Step 3: Evaluate predictions using the test matrix
# Explanation:
# - Use the evaluate() function to compute performance of this method.
# - MAE (Mean Absolute Error) and RMSE (Root Mean Square Error) show how close
#   the predicted ratings are to the true ratings in the test set.

MAE_solution2, RMSE_solution2 = evaluate(test_ds, predicted_ds2)

# Print the final evaluation metrics for Method 2 (Item Mean)
print(f"Method 2 — Item Mean MAE: {MAE_solution2:.6f}, RMSE: {RMSE_solution2:.6f}")



Method 2 — Item Mean MAE: 0.796120, RMSE: 1.001314


In [8]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution2, RMSE_solution2))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.7961203951019012, RMSE: 1.001314210158761


# Your Solution for Method 3

In [9]:
# Write your code here for Method 3
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution3 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution3 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.

# --------------------------
# Method 3: User-KNN Collaborative Filtering
# --------------------------

# Hyperparameters
K = 50          # Number of top similar users to consider for prediction
EPSILON = 1e-9  # Small constant to prevent division by zero

# Step 1: Compute each user's average rating from the training set
# Explanation:
# - user_counts: Number of items rated by each user
# - user_sums: Sum of ratings for each user
# - user_means: Average rating per user (handle zero ratings with EPSILON)
user_counts = (train_ds > 0).sum(axis=1)
user_sums = train_ds.sum(axis=1)
user_means = np.where(
    user_counts > 0,
    user_sums / (user_counts + EPSILON),
    0.0
)

# Step 2: Center the ratings by subtracting user means
# Explanation:
# - Only subtract means for non-zero entries (actual ratings)
# - Resulting matrix has zero-centered values, similar to Pearson correlation logic
centered_train = (train_ds - user_means[:, None]) * (train_ds > 0)

# Step 3: Compute user-user similarity using cosine similarity on centered ratings
# Explanation:
# - Cosine similarity approximates Pearson correlation when data is mean-centered
from sklearn.metrics.pairwise import cosine_similarity
user_sim = cosine_similarity(centered_train)

# Step 4: Make predictions only for the positions present in the test set
# Explanation:
# - Initialize prediction matrix
# - For each (user, item) in test set:
#   a) Identify other users (neighbours) who rated the item
#   b) From those, pick top K similar users
#   c) Compute a weighted sum of (rating - mean) from neighbours
#   d) Add back the target user’s mean to get final prediction

predicted_ds3 = np.zeros_like(train_ds)

# Get all (user, item) pairs with non-zero ratings in test set
test_positions = np.argwhere(test_ds > 0)

# Loop over all test set entries
for u, i in test_positions:
    # Find neighbours (other users who rated item i)
    neighbours = np.where(train_ds[:, i] > 0)[0]

    if neighbours.size == 0:
        # Fallback: no neighbours, use user's own mean rating
        pred = user_means[u]
    else:
        # Compute similarity scores between user u and all neighbours
        sims = user_sim[u, neighbours]

        # Select top K most similar neighbours
        top_k_ids = neighbours[np.argsort(sims)[-K:]]

        # Fetch similarity scores, ratings, and means for top-K neighbours
        sims_k = user_sim[u, top_k_ids]
        ratings_k = train_ds[top_k_ids, i]
        means_k = user_means[top_k_ids]

        # Compute numerator and denominator for weighted sum
        num = np.sum(sims_k * (ratings_k - means_k))
        den = np.sum(np.abs(sims_k)) + EPSILON  # avoid division by 0

        # Final prediction: add weighted adjustment to user’s own mean
        pred = user_means[u] + num / den

    # Save prediction in matrix
    predicted_ds3[u, i] = pred

# Step 5: Evaluate prediction performance on test set
MAE_solution3, RMSE_solution3 = evaluate(test_ds, predicted_ds3)
print(f"Method 3 — User-KNN CF MAE: {MAE_solution3:.6f}, RMSE: {RMSE_solution3:.6f}")


Method 3 — User-KNN CF MAE: 0.721260, RMSE: 0.927265


In [10]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution3, RMSE_solution3))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.7212601962692197, RMSE: 0.9272652293442558


# Your Solution for Method 4

In [11]:
# Write your code here for Method 4
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution4 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution4 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.

# --------------------------
# Method 4: Item-KNN Collaborative Filtering
# --------------------------

# 0. Define neighbourhood size (K) for item-based similarity
K_item = 20  # We consider top 20 most similar items for each prediction

# 1. Compute each item's mean rating from the training matrix
# Explanation:
# - item_counts: how many users rated each item (column-wise)
# - item_sums: total sum of ratings for each item
# - item_means: average rating for each item (avoid division by zero using EPSILON)
item_counts = (train_ds > 0).sum(axis=0)
item_sums = train_ds.sum(axis=0)
item_means = np.where(
    item_counts > 0,
    item_sums / (item_counts + EPSILON),
    0.0
)

# 2. Transpose the training data to item-user format and mean-center it
# Explanation:
# - item_train: rows are items, columns are users
# - mask_item: identifies where valid ratings exist
# - centered_item: subtract item mean only where ratings exist
item_train = train_ds.T  # shape: (n_items, n_users)
mask_item = item_train > 0
centered_item = (item_train - item_means[:, None]) * mask_item

# 3. Compute item-item similarity matrix using centered-cosine similarity
# Explanation:
# - cosine_similarity acts like Pearson correlation when data is mean-centered
from sklearn.metrics.pairwise import cosine_similarity
item_sim = cosine_similarity(centered_item)  # shape: (n_items, n_items)

# 4. Predict ratings for test set positions only
predicted_ds4 = np.zeros_like(train_ds)

# Get (user, item) pairs from the test matrix where ratings exist
test_positions = np.argwhere(test_ds > 0)

# Loop over each test position to generate predictions
for u, i in test_positions:
    # Step 4a: Identify items rated by the user u
    neighbours = np.where(train_ds[u, :] > 0)[0]  # item indices

    if neighbours.size == 0:
        # Fallback: if user hasn't rated any items, return average rating of item i
        pred = item_means[i]
    else:
        # Step 4b: Get similarities between target item i and its neighbours
        sims = item_sim[i, neighbours]

        # Step 4c: Select top K most similar items rated by the user
        top_k_ids = neighbours[np.argsort(sims)[-K_item:]]

        # Step 4d: Extract similarity values and corresponding ratings
        sims_k = item_sim[i, top_k_ids]
        ratings_k = train_ds[u, top_k_ids]
        means_k = item_means[top_k_ids]

        # Step 4e: Compute prediction using weighted sum of rating deviations
        num = np.sum(sims_k * (ratings_k - means_k))          # numerator
        den = np.sum(np.abs(sims_k)) + EPSILON               # denominator to avoid zero

        # Step 4f: Final predicted rating is item's mean plus weighted adjustment
        pred = item_means[i] + num / den

    # Save prediction in the matrix
    predicted_ds4[u, i] = pred

# 5. Evaluate predictions using MAE and RMSE
MAE_solution4, RMSE_solution4 = evaluate(test_ds, predicted_ds4)
print(f"Method 4 — Item-KNN CF MAE: {MAE_solution4:.6f}, RMSE: {RMSE_solution4:.6f}")


Method 4 — Item-KNN CF MAE: 0.702684, RMSE: 0.901183


In [12]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution4, RMSE_solution4))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.702683844709954, RMSE: 0.9011829140124539


# Your Solution for Method 5

In [13]:
# Write your code here for Method 5
# You are required to implement the required solution 1 here. 
# Then, evaluate your implementation by predicting the ratings in the test set (test_ds).
# Finally, save the corresponding MAE and RMSE of your implementation 
# into the following defined corresponding variable. 

MAE_solution5 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.
RMSE_solution5 = 0 # 0 is an intial value, you need to update this with the actual perofrmance of your implementation.

# --------------------------
# Method 5: Hybrid CF
# --------------------------

# 1. Choose a blending weight for combining user-based and item-based predictions
# -------------------------------------------------------------------------------
# lambda_val determines the balance:
# - λ = 1.0 means use 100% of user-based predictions (Method 3)
# - λ = 0.0 means use 100% of item-based predictions (Method 4)
# - λ = 0.5 gives equal weight to both (a true hybrid)
lambda_val = 0.5  # You can tune this value to minimize MAE/RMSE

# 2. Combine the predictions from Method 3 (User-KNN CF) and Method 4 (Item-KNN CF)
# ----------------------------------------------------------------------------------
# - predicted_ds3: the prediction matrix from user-based CF
# - predicted_ds4: the prediction matrix from item-based CF
# - The hybrid prediction is a weighted sum of both
predicted_ds5 = lambda_val * predicted_ds3 + (1 - lambda_val) * predicted_ds4

# 3. Evaluate the hybrid predictions against the test set
# -------------------------------------------------------
# - Uses the standard `evaluate()` function to compute MAE and RMSE
# - Helps assess whether blending improves performance over individual methods
MAE_solution5, RMSE_solution5 = evaluate(test_ds, predicted_ds5)

# 4. Print the performance metrics for the hybrid model
print(f"Method 5 — Hybrid CF (λ={lambda_val}) MAE: {MAE_solution5:.6f}, RMSE: {RMSE_solution5:.6f}")


Method 5 — Hybrid CF (λ=0.5) MAE: 0.703871, RMSE: 0.903897


In [14]:
# Please don't change this cell

print("===================== The MAE and RMSE of Your Implementation =====================")
print("MAE: {}, RMSE: {}" .format(MAE_solution5, RMSE_solution5))

===================== The MAE and RMSE of Your Implementation =====================
MAE: 0.7038714282908554, RMSE: 0.9038968442080455


In [15]:
# -----------------------------------------------------------------------------
# Lambda Optimization: Find the best weight (λ) for Hybrid CF (User-KNN + Item-KNN)
# -----------------------------------------------------------------------------

# Initialize tracking variables to store the best performance seen so far
# float('inf') ensures that any valid MAE will be lower than this initial value
best_mae, best_rmse, best_lambda = float('inf'), float('inf'), None

# Try λ values from 0 to 1 in steps of 0.1 (i.e., [0.0, 0.1, 0.2, ..., 1.0])
# This determines the weight given to the user-based prediction matrix (predicted_ds3)
# and (1 - λ) will be the weight for the item-based prediction matrix (predicted_ds4)
for lam in np.linspace(0, 1, 11):
    
    # Generate hybrid predictions using the current lambda
    pred = lam * predicted_ds3 + (1 - lam) * predicted_ds4

    # Evaluate the hybrid prediction on the test set
    mae, rmse = evaluate(test_ds, pred)

    # Check if this λ gives a lower MAE than previous best
    if mae < best_mae:
        # Update the best-known MAE, RMSE, and corresponding λ
        best_mae, best_rmse, best_lambda = mae, rmse, lam

# Print out the best-performing λ value and its corresponding evaluation metrics
print(f"Best λ={best_lambda}: MAE={best_mae:.6f}, RMSE={best_rmse:.6f}")


Best λ=0.2: MAE=0.701248, RMSE=0.899746
